# Train Card Strength Model

This notebook focuses on training and evaluating regression models to predict MTG card power levels. It explores both tabular (XGBoost) and text-based (MLP) approaches, ultimately combining them into an ensemble for superior performance.

**Objectives:**
- Prepare the dataset for training by splitting and scaling features.
- Train an XGBoost model on engineered tabular features.
- Train a Multi-Layer Perceptron (MLP) on dense text embeddings.
- Evaluate model accuracy using MAE and RMSE metrics.
- Create an ensemble model to leverage both tabular and semantic data.

### 1. Environment Setup

- Import core data science and machine learning libraries.
- Set global random seed for reproducibility.

In [109]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

### 2. Configuration

- Define input data path and target model output path.
- Set the random seed to `1893`.

In [110]:
INPUT_DATA_PATH = "data/card_strength_dataset.parquet"
OUTPUT_MODEL_PATH = "card_strength_model.pth"
SEED = 1893

### 3. Data Loading

- Load the processed card strength dataset from Parquet.
- Initialize the global seed for library-specific operations.

In [111]:
np.random.seed(SEED)
df = pd.read_parquet(INPUT_DATA_PATH)
print(f"shape: {df.shape}")
df.head()

### 4. Feature Preparation

- Extract embedding, numerical, and boolean columns.
- Handle missing values in numerical features with a sentinel value (-999).
- Create the feature matrix `X` and target vector `y`.

In [112]:
embedding_cols = [col for col in df.columns if col.startswith('embed_')]
embedding_features = df[embedding_cols].values
boolean_cols = [col for col in df.columns if col.startswith('bool_')]
boolean_features = df[boolean_cols].values
numerical_cols = [col for col in df.columns if col.startswith('num_')]
numerical_features = df[numerical_cols].fillna(-999).values

X = np.hstack((embedding_features, numerical_features, boolean_features))
y = df['log10_power_score'].values
print(f"Final Feature Matrix (X) ready! Shape: {X.shape}")

### 5. Correlation Analysis

- Calculate Pearson correlation between features and the target power score.
- Plot the top positive and negative correlations to validate feature engineering.

In [113]:
feature_names = embedding_cols + numerical_cols + boolean_cols
correlations = df[feature_names + ['log10_power_score']].corr()['log10_power_score'].drop('log10_power_score')
top_correlations = pd.concat([correlations.nsmallest(10), correlations.nlargest(10)]).sort_values()

plt.figure(figsize=(12, 8))
sns.barplot(x=top_correlations.values, y=top_correlations.index)
plt.title('Top Correlations with Power Score', fontsize=16)
plt.show()

### 6. Data Splitting

- Split data into Training (60%), Validation (20%), and Test (20%) sets.
- Ensure stratification and reproducibility using the defined seed.

In [115]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=SEED)
print(f"Training data: {X_train.shape[0]} cards")

### 7. XGBoost Training

- Train a tuned XGBoost regressor using early stopping.
- Optimize for learning rate, depth, and subsampling to prevent overfitting.
- Visualize feature importance to identify key performance drivers.

In [117]:
xgbr_tuned = xgb.XGBRegressor(
    n_estimators=3000, learning_rate=0.03, max_depth=4,
    subsample=0.8, colsample_bytree=0.6, early_stopping_rounds=50,
    random_state=SEED, n_jobs=-1
)
xgbr_tuned.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=100)
y_xgbr_preds = xgbr_tuned.predict(X_test)
print(f"XGBoost MAE: {mean_absolute_error(y_test, y_xgbr_preds):.3f}")

### 8. MLP (Text-based) Training

- Isolate embedding features for a Multi-Layer Perceptron (Neural Network).
- Scale embeddings to zero mean and unit variance.
- Train the MLP to capture semantic nuances in card text that tabular features might miss.

In [121]:
num_embeds = len(embedding_cols)
X_train_mlp = X_train_val[:, :num_embeds]
X_test_mlp  = X_test[:, :num_embeds]
scaler = StandardScaler()
X_train_mlp_scaled = scaler.fit_transform(X_train_mlp)
X_test_mlp_scaled = scaler.transform(X_test_mlp)

mlp_tuned = MLPRegressor(
    hidden_layer_sizes=(512, 256, 64), alpha=0.5, early_stopping=True,
    random_state=SEED, verbose=False
)
mlp_tuned.fit(X_train_mlp_scaled, y_train_val)
y_mlp_preds = mlp_tuned.predict(X_test_mlp_scaled)
print(f"MLP MAE: {mean_absolute_error(y_test, y_mlp_preds):.3f}")

### 9. Ensemble and Evaluation

- Create a weighted ensemble of XGBoost and MLP predictions.
- Evaluate the ensemble's performance on the test set.
- Plot absolute errors and residuals to analyze model bias and variance.

In [122]:
ensemble_preds = (y_xgbr_preds + y_mlp_preds) / 2.0
print(f"Ensemble MAE: {mean_absolute_error(y_test, ensemble_preds):.3f}")
print(f"Ensemble RMSE: {np.sqrt(mean_squared_error(y_test, ensemble_preds)):.3f}")